# Generalized Additive Models

**DSAI 621 — Week 5.** A generalized additive model (GAM) replaces each linear term
$\beta_jx_{ij}$ of a linear model with a function $f_j(x_{ij})$ (ISL 7.7):
$$y_i = \beta_0 + f_1(x_{i1}) + f_2(x_{i2}) + \dots + f_p(x_{ip}) + \varepsilon_i.$$
Each $f_j$ can be a smooth curve, a straight line, or a set of level effects for a categorical
predictor, and the terms are simply added. In this notebook you will:

1. fit a GAM with a smooth term for each numeric predictor, and read the fitted curves;
2. fit a GAM with a **mix of terms** (smooth, linear and categorical), and decide when a straight line
   is enough;
3. see how the smoothing parameter $\lambda$ controls each term, and choose each $\lambda$ by
   generalized cross-validation;
4. fit a **logistic GAM** for a binary response.

All of the code is written for you: run each cell, read what it does, and change it to try
things out. Short exercises marked **E1**, **E2**, … ask you to stop and look at what you just
made, and answer in words. Exercises marked **T1**, **T2**, … ask you to change a number in a cell,
rerun it, and explain what changed. Each one says which cells to rerun and whether to change the
number back afterwards.

> **A word about tools: for GAMs, use R's `mgcv` if you can.** Python is not the best tool for
> GAMs, or for splines in general. The R package **`mgcv`** is by far the better option. Simon Wood
> has developed it for more than twenty years, and it covers the sheer breadth of this topic:
> many kinds of smoother, smooth interactions, many response distributions, smoothing parameters
> chosen by REML or GCV with fast, reliable optimization rather than a grid search, and confidence
> intervals and tests that account for the smoothing. His book, *Generalized Additive Models: An
> Introduction with R*, documents it. Nothing in Python comes close. We use
> [**pyGAM**](https://pygam.readthedocs.io), a small package that is enough to see the ideas, and
> the one ISL's Python labs use (ISL Lab 7.8.3), so you can follow those labs and their exercises with
> the same functions. It is maintained by a few volunteers: as of this course it requires an older version of SciPy, and its
> own `summary()` warns that its p-values are unreliable. `statsmodels` also has GAMs, but with
> fewer features, and keeping even `statsmodels` going is already a lot for an open-source
> project. If you need GAMs for real work, use `mgcv`.

**pyGAM's smooth terms.** ISL's smoothing spline puts a knot at every $x_i$ and penalizes
$\int f''(t)^2\,dt$. pyGAM's `s()` term is a close cousin called a **P-spline**: 20 cubic B-spline
basis functions spread over the range of $x$, with a penalty $\lambda$ times the sum of squared
second differences of neighbouring coefficients. The idea is the same. $\lambda = 0$ gives an
unpenalized regression spline, and as $\lambda \to \infty$ the term is forced towards a straight line.
But pyGAM's $\lambda$ is on a different scale from ISL's, so compare fits by their **effective degrees
of freedom** (pyGAM's "EDoF", the trace of the smoother, as in the splines lab), not by $\lambda$.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pygam import LinearGAM, LogisticGAM, f, l, s
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

warnings.filterwarnings("ignore", message="KNOWN BUG")   # pyGAM's pop-up warning about its own p-values

# random number generator for reproducibility
rng = np.random.default_rng(621)

### The data and the helper functions

The data are simulated, so we know the true terms and can check the fits against them. There are
two numeric predictors, $x_1$ and $x_2$, uniform on $[0, 1]$, and a categorical predictor $g$ with
levels 0, 1 and 2 (probabilities 0.5, 0.3 and 0.2):
$$y_i = 2 + \underbrace{\sin(2\pi x_{i1})}_{f_1:\ \text{a curve}} + \underbrace{1.5\,x_{i2}}_{f_2:\ \text{a line}} + \underbrace{f_3(g_i)}_{\text{level effects}} + \varepsilon_i, \qquad \varepsilon_i \sim N(0, 0.5^2),$$
with $f_3(0) = -0.5$, $f_3(1) = 0$ and $f_3(2) = 1$. So one term really is curved, one really is
linear, and one is categorical. pyGAM refers to predictors by column number: `s(0)` is a smooth term
in column 0 ($x_1$), `l(1)` a linear term in column 1 ($x_2$), and `f(2)` a factor term in column 2
($g$).

We fit on 300 observations and measure the test MSE on 5000 more. The noise variance is
$0.5^2 = 0.25$, so no model can have a test MSE much below 0.25.

The helpers:

- `term_edof(gam)` returns the effective degrees of freedom of each term of a fitted model.
- `plot_terms(gam, X, truths)` plots each fitted term with its 95% confidence band and the true
  term. pyGAM doesn't center its terms, so both are **centered** (shifted to average zero over the
  training data) before plotting: only the shape of a term means anything, because a constant can
  move freely between a term and the intercept.

In [ ]:
# The true terms for f1 and f2
# f1 is smooth
def f1_true(x):
    return np.sin(2 * np.pi * x)

# f2 is linear
def f2_true(x):
    return 1.5 * x

# The true group effect for the factor term g
# this is essentially f3
GROUP_EFFECT = np.array([-0.5, 0.0, 1.0])

def make_data(n, rng, sigma=0.5):
    "Columns x1, x2, g of X, and y."
    x1, x2 = rng.uniform(0, 1, n), rng.uniform(0, 1, n)
    g = rng.choice(3, size=n, p=[0.5, 0.3, 0.2])
    y = 2 + f1_true(x1) + f2_true(x2) + GROUP_EFFECT[g] + rng.normal(0, sigma, n)
    return np.column_stack([x1, x2, g]), y

def term_edof(gam):
    "Effective degrees of freedom of each term (the intercept is left out)."
    per_coef = gam.statistics_["edof_per_coef"]
    return [per_coef[gam.terms.get_coef_indices(i)].sum() for i in range(len(gam.terms))
            if not gam.terms[i].isintercept]

# Mean squared error of the predictions of a GAM on a dataset (X, y).
def mse(gam, X, y):
    return np.mean((gam.predict(X) - y) ** 2)

# Extract 
def plot_terms(gam, X, truths, names=("x1", "x2", "g"), ylabel="f"):
    "Each fitted term (centered, with a 95% band) and the true term (centered)."
    fig, axes = plt.subplots(1, len(truths), figsize=(4.3 * len(truths), 3.6))
    for i, (ax, truth, name) in enumerate(zip(axes, truths, names)):
        shift = gam.partial_dependence(term=i, X=X).mean()       # average of the term over the data
        if callable(truth):                                      # a numeric term: plot a curve
            XX = gam.generate_X_grid(term=i)
            t = XX[:, gam.terms[i].feature]
            pdep, band = gam.partial_dependence(term=i, X=XX, width=0.95)
            ax.plot(t, pdep - shift, lw=2, label="fitted")
            ax.fill_between(t, band[:, 0] - shift, band[:, 1] - shift, alpha=0.25, label="95% band")
            ax.plot(t, truth(t) - truth(X[:, i]).mean(), "k:", lw=2, label="true")
        else:                                                    # a factor term: one value per level
            XX = np.zeros((3, X.shape[1])); XX[:, i] = [0, 1, 2]
            pdep, band = gam.partial_dependence(term=i, X=XX, width=0.95)
            ax.errorbar([0, 1, 2], pdep - shift, yerr=[pdep - band[:, 0], band[:, 1] - pdep],
                        fmt="o", ms=7, capsize=5, label="fitted, with 95% band")
            ax.plot([0, 1, 2], truth - truth[X[:, i].astype(int)].mean(), "kx", ms=10, mew=2, label="true")
            ax.set_xticks([0, 1, 2])
        ax.axhline(0, color="grey", lw=0.6)
        kind = {"spline_term": "s", "linear_term": "l", "factor_term": "f"}[str(gam.terms[i])]
        ax.set(xlabel=name, ylabel=f"{ylabel}({name})", title=f"{kind}({name})  (EDoF {term_edof(gam)[i]:.1f})")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

# Make the training and test datasets
# Test data is larger to get a more stable estimate of the test error
X, y = make_data(300, rng)
X_test, y_test = make_data(5000, rng)
print("training data:", X.shape, " test data:", X_test.shape)

---
## 1. A GAM with smooth terms

Start by treating both numeric predictors the same way, with a smooth term for each, and the group
as a factor:
$$y = \beta_0 + f_1(x_1) + f_2(x_2) + f_3(g) + \varepsilon.$$
`LinearGAM` means a GAM for a numeric response with normal errors (a "linear" model in the sense of
DSAI 601: least squares, with a penalty). For now every term uses pyGAM's default, $\lambda = 0.6$.
`summary()` lists each term's $\lambda$, its rank (the number of basis functions) and its EDoF.

In [ ]:
# s() is a spline term, f() is a factor term, l() is a linear term
gam_smooth = LinearGAM(s(0) + s(1) + f(2)).fit(X, y)
gam_smooth.summary()
print(f"test MSE: {mse(gam_smooth, X_test, y_test):.4f}")
plot_terms(gam_smooth, X, [f1_true, f2_true, GROUP_EFFECT])

**E1.** Compare each fitted term with the true one. Which term looks least like the truth, and
what is wrong with it? Look at the EDoF column in the summary: how many degrees of freedom does each
smooth term use, and is that sensible for a sine wave? For a straight line?

<details>
<summary>Show answer: E1</summary>

**E1. Answer.** The $x_2$ term looks least like the truth. The true $f_2$ is a straight line, but
the fitted term wiggles around it. The $x_1$ term follows the sine wave well, except near $x_1 = 1$,
where it flattens out. The group effects are close. The smooth terms use 12.2 and 10.7 EDoF. A
straight line needs only 1, so 10.7 is mostly spent fitting noise. Even the sine wave doesn't need
12.2: Section 3 finds that about 7 is better. The default $\lambda = 0.6$ is too small for both terms.

</details>

**E2.** The bands get wider at the edges of the $x_1$ and $x_2$ plots. Why? (Think back to the
standard errors in the splines lab.)

<details>
<summary>Show answer: E2</summary>

**E2. Answer.** Near the edges there are fewer observations close by, and they are all on one
side, so the fit there rests on less information. In the splines lab, the standard error of a fitted
spline was largest at the boundary for the same reason: the leverages are highest there. The same
thing happens to each term of a GAM.

</details>

**T1.** In `make_data(300, rng)` in the data cell, change 300 to 3000, then rerun the data cell
and the cell above. What happens to the bands, to the wiggles in the $x_2$ term, and to the test MSE?
Change it back to 300 afterwards and rerun both cells.

<details>
<summary>Show answer: T1</summary>

**T1. Answer.** With 3000 observations the bands are much narrower: their average width falls from
about 0.37 to 0.15. The fitted $x_1$ term is closer to the truth, and its largest error falls from 0.31
to 0.11. The test MSE falls from 0.262 to 0.254, close to the floor of 0.25. The $x_2$ term still
wiggles, and its EDoF actually *rises*, to 15.2 (and $x_1$'s to 16.2). With $\lambda$ held at 0.6, the
penalty stays the same size while the RSS grows with $n$, so the penalty matters less and every term
gets more flexible. The same $\lambda$ means a different amount of smoothing at a different $n$. This
is another reason to choose $\lambda$ from the data rather than fixing it.

</details>

---
## 2. A GAM with a mix of terms

A GAM doesn't have to make every term a curve. ISL 7.7.1 fits wage with smooth terms in year and
age and a step for each level of education. In pyGAM:

- `s(j)`: a smooth term (a penalized spline);
- `l(j)`: a linear term, $\beta_jx_j$, exactly as in linear regression (1 df);
- `f(j)`: a factor term, one effect per level, like dummy variables.

The true $f_2$ is a straight line, so a smooth term spends degrees of freedom on wiggles that are
only noise. The cell below fits the model again with a linear term for $x_2$ and compares the two.

In [ ]:
# Now fit a GAM with a linear term for x2 instead of a spline term
gam_mixed = LinearGAM(s(0) + l(1) + f(2)).fit(X, y)

# Make a table comparing the two GAMs
rows = []
for name, gam in [("s(0) + s(1) + f(2)", gam_smooth), ("s(0) + l(1) + f(2)", gam_mixed)]:
    e = term_edof(gam)
    rows.append({"model": name, "EDoF of x1 term": e[0], "EDoF of x2 term": e[1],
                 "EDoF of g term": e[2], "total EDoF": gam.statistics_["edof"],
                 "GCV": gam.statistics_["GCV"], "test MSE": mse(gam, X_test, y_test)})
display(pd.DataFrame(rows).set_index("model").round(3))

# Now plot the fitted terms of the mixed GAM
plot_terms(gam_mixed, X, [f1_true, f2_true, GROUP_EFFECT])

**E3.** How many degrees of freedom does the linear term use, and how many did the smooth $x_2$
term use? Which model has the lower test MSE, and why? The GCV column estimates test error from the
training data alone (it is the LOOCV shortcut from the splines lab, with every leverage replaced by
their average). Does it pick the same model?

<details>
<summary>Show answer: E3</summary>

**E3. Answer.** The linear term uses 0.96 EDoF: one slope, shrunk a little because pyGAM also
applies its $\lambda = 0.6$ as a small ridge penalty on linear terms. The smooth $x_2$ term used 10.7.
The total falls from 24.8 to 14.9. The mixed model has the lower test MSE (0.259 against 0.262),
because it doesn't spend 10 extra degrees of freedom fitting noise, which only adds variance. GCV agrees
(0.257 against 0.276), and it uses only the training data.

</details>

**E4.** The factor term's EDoF is a little under 2, although $g$ has 3 levels. Why 2 and not 3,
and why a little under 2? (Hint: pyGAM penalizes factor terms too, by default with the same
$\lambda = 0.6$.)

<details>
<summary>Show answer: E4</summary>

**E4. Answer.** Three levels need only 2 free values once the model has a constant: the overall
level belongs to the intercept (in pyGAM it is actually booked to the first smooth term, which is why
the intercept shows 0 EDoF in the summary). This is the same reason that 601's dummy coding uses $K - 1$
dummies. It is a little under 2 because pyGAM penalizes factor terms too. The penalty shrinks the level
effects towards each other, as ridge regression shrinks coefficients, so the term has slightly fewer
effective degrees of freedom than its 2 free values.

</details>

**E5.** With real data you don't know which terms are linear. What is one way you can decide, using
what you've seen in this notebook so far.

<details>
<summary>Show answer: E5</summary>

**E5. Answer.** (1) Fit the predictor both ways, `s(j)` and `l(j)`, and compare GCV, or a validation
or cross-validation error, as in E3. (2) Give the term a smooth and let a grid search choose its own
$\lambda_j$. If GCV chooses a very large $\lambda_j$ and the term's EDoF comes out near 1, a straight
line is enough. (3) Plot the smooth term with its band, and see whether a straight line fits inside the
band everywhere. (Option 2 is what you'll see in Section 3.3.)

</details>

---
## 3. Setting $\lambda$

Each smooth term has its own $\lambda_j$. The penalty for the whole model is
$\sum_j\lambda_j \times (\text{roughness of } f_j)$, so a large $\lambda_j$ makes term $j$ smoother
and a small one lets it follow the data more closely. This is the same dial as in the splines lab,
one per term.

### 3.1 One term's dial

The cell below refits the mixed model with four values of $\lambda$ for the $x_1$ term, keeping
everything else the same, and plots the $x_1$ term for each.

In [ ]:
# For values of lambda to test on the spline term
lams = [1e-3, 0.6, 1e3, 1e6]

# Loop through lambdas and create a table of results and a dictionary of fitted GAMs
rows, fits = [], {}
for lam in lams:
    gam = LinearGAM(s(0, lam=lam) + l(1) + f(2)).fit(X, y)
    rows.append({"lambda for x1": lam, "EDoF of x1 term": term_edof(gam)[0],
                 "GCV": gam.statistics_["GCV"], "test MSE": mse(gam, X_test, y_test)})
    fits[lam] = gam
display(pd.DataFrame(rows).set_index("lambda for x1").rename(index="{:g}".format).round(3))

fig, ax = plt.subplots(figsize=(7, 4))
XX = fits[lams[0]].generate_X_grid(term=0)
for lam, ls in zip(lams, ["-", "--", "-.", (0, (1, 1))]):
    gam = fits[lam]
    shift = gam.partial_dependence(term=0, X=X).mean()
    ax.plot(XX[:, 0], gam.partial_dependence(term=0, X=XX) - shift, ls=ls, lw=2,
            label=f"lambda = {lam:g} (EDoF {term_edof(gam)[0]:.1f})")
ax.plot(XX[:, 0], f1_true(XX[:, 0]) - f1_true(X[:, 0]).mean(), "k:", lw=1.5, label="true")
ax.set(xlabel="x1", ylabel="f(x1)", title="The x1 term for four values of lambda")
ax.legend(fontsize=8)
plt.show()

**E6.** Describe the $x_1$ term at each $\lambda$. Which is closest to the truth, and which two
are clearly wrong, and how? What does the term become as $\lambda \to \infty$? (Compare the EDoF at
$\lambda = 10^6$ with the linear term's in Section 2.)

<details>
<summary>Show answer: E6</summary>

**E6. Answer.** At $\lambda = 0.001$ (19.7 EDoF) the term is very wiggly and chases the noise, with
a spike near $x_1 = 0$: test MSE 0.270. At $\lambda = 0.6$ (12.0 EDoF) it is close to the truth: test
MSE 0.259, the best of the four. At $\lambda = 1000$ (3.3 EDoF) it is too smooth: the wave is
flattened into a gentle S, and the test MSE is 0.364. At $\lambda = 10^6$ it is a **straight line**,
and the test MSE is 0.455. As $\lambda \to \infty$, the penalty on second differences forces the term
to have no curvature, so it becomes a line, just as for ISL's smoothing spline. Its EDoF is 2.2, not
about 1 like the linear term, because pyGAM books the model's overall constant to the first smooth term
rather than to the intercept. So 1 of the 2.2 is that constant, and about 1 is the slope.

</details>

### 3.2 The same $\lambda$ doesn't mean the same smoothness

In Section 1 both smooth terms had the default $\lambda = 0.6$. Look back at their EDoF in the
summary. The same $\lambda$ gave the two terms different degrees of freedom, because how much a
given $\lambda$ smooths depends on the data for that term: how the $x$ values are spread out, how
many basis functions there are, and (for ISL's smoothing spline) the units of $x$. That is why you
should think about each term's **EDoF**, not its $\lambda$, and why each term needs its own
$\lambda_j$.

### 3.3 Choosing $\lambda$ by GCV

Trying every combination of $\lambda_j$ by hand isn't practical, so `gridsearch` fits the model for
a grid of values and keeps the one with the lowest GCV. There are two ways to use it:

- **one shared $\lambda$** (pyGAM's default): the same value for every term, tried at 11 values from
  $10^{-3}$ to $10^3$;
- **a separate $\lambda_j$ for each term**: pass one grid per term, and pyGAM tries every
  combination. With 7 values for each of 3 terms that is $7^3 = 343$ fits.

The cell below does both, for the all-smooth model of Section 1. (The `R` package, `mgcv`, doesn't need a grid: it
optimizes all the $\lambda_j$ at once, which is much faster when there are many terms.)

In [ ]:
# Fit a GAM with same lambda for all terms, using GCV to choose the lambda
gam_shared = LinearGAM(s(0) + s(1) + f(2)).gridsearch(X, y, progress=False)

# Create a grid of values for lambda to test for each term, 
# and fit a GAM with a different lambda for each term, using GCV to choose the lambdas
grid = np.logspace(-3, 3, 7)
gam_each = LinearGAM(s(0) + s(1) + f(2)).gridsearch(X, y, lam=[grid, grid, grid], progress=False)

# Make a table comparing the three GAMs
# Plot the fitted terms of the GAM with a different lambda for each term
rows = []
for name, gam in [("default, lambda = 0.6", gam_smooth), ("one shared lambda, by GCV", gam_shared),
                  ("a lambda for each term, by GCV", gam_each)]:
    lam = [float(v[0]) for v in gam.lam]
    e = term_edof(gam)
    rows.append({"model": name, "lambda x1": lam[0], "lambda x2": lam[1], "lambda g": lam[2],
                 "EDoF x1": e[0], "EDoF x2": e[1], "EDoF g": e[2],
                 "GCV": gam.statistics_["GCV"], "test MSE": mse(gam, X_test, y_test)})
display(pd.DataFrame(rows).set_index("model").round(3))
plot_terms(gam_each, X, [f1_true, f2_true, GROUP_EFFECT])

**E7.** What $\lambda$ does the shared search choose, and what does it do to the two smooth terms?
What does the search with a separate $\lambda_j$ for each term choose for the $x_2$ term, and what
does its EDoF tell you? Compare the test MSEs with the mixed model of Section 2.

<details>
<summary>Show answer: E7</summary>

**E7. Answer.** The shared search chooses $\lambda = 3.98$ for every term. That smooths both terms
more than the default (EDoF 8.9 for $x_1$ and 7.5 for $x_2$), but one value has to suit both, so $x_2$
still gets 7.5 EDoF for a straight line. With a separate $\lambda_j$ for each term, the search chooses
$\lambda = 1000$ for $x_2$, the largest value in the grid, and its EDoF falls to 2.2. GCV has found by
itself that $x_2$ is (nearly) linear. It also chooses $\lambda = 10$ for $x_1$, which gives 7.4 EDoF,
smoother than the default. Test MSE: 0.262 for the default, 0.260 for the shared $\lambda$, and 0.256
for the separate $\lambda_j$. That is better than the mixed model of Section 2 (0.259), which used the
right form for $x_2$ but kept the default $\lambda$ for $x_1$.

</details>

**T2.** In the cell above, change `grid = np.logspace(-3, 3, 7)` to `np.logspace(-3, 6, 10)`
and rerun it. What does the search choose for the $x_2$ term now, and what does that term become?
How many fits does the search do now? Change it back afterwards and rerun.

<details>
<summary>Show answer: T2</summary>

**T2. Answer.** The search again chooses the largest value in the grid for $x_2$, now
$\lambda = 10^6$. The term's EDoF is 1.10, so it is essentially the straight line that `l(1)` would
give. The test MSE stays at 0.256 and GCV falls slightly, to 0.248. When the chosen $\lambda$ is at the
edge of the grid, extend the grid: the best value may be beyond it. Now there are 10 values for each
term, so the search does $10^3 = 1000$ fits instead of 343. This is why grids over every combination
don't scale to many terms, and why `mgcv` optimizes instead.

</details>

---
## 4. A logistic GAM

For a binary response, a logistic GAM puts the additive model on the log-odds (ISL 7.7.2):
$$\log\frac{p(x)}{1 - p(x)} = \beta_0 + f_1(x_1) + f_2(x_2) + f_3(g).$$
`LogisticGAM` fits it by penalized maximum likelihood, and the terms are read on the log-odds scale.
The cell below simulates a binary response with true log-odds
$$-0.5 + 2\sin(2\pi x_1) + 3(x_2 - 0.5) + f_3(g),$$
so $x_1$ is curved, $x_2$ is linear and $g$ is categorical. It fits a logistic GAM with a mix of terms
(choosing a $\lambda_j$ for each term by a grid search) and an ordinary logistic regression that is
linear in $x_1$ and $x_2$, then compares them on a large test set by the misclassification rate and
the log loss (the average negative log-likelihood; smaller is better).

In [ ]:
# Make a binary response variable, with the same true terms as the continuous response variable
def make_binary(n, rng):
    Xb, _ = make_data(n, rng)
    logit = -0.5 + 2 * f1_true(Xb[:, 0]) + 3 * (Xb[:, 1] - 0.5) + GROUP_EFFECT[Xb[:, 2].astype(int)]
    yb = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)
    return Xb, yb

# Create training and test datasets for the binary response variable
Xb, yb = make_binary(500, rng)
Xb_test, yb_test = make_binary(5000, rng)

# Fit a logistic GAM with a spline term for x1, a linear term for x2, and a factor term for g
grid_b = np.logspace(-3, 3, 7)
gam_logit = LogisticGAM(s(0) + l(1) + f(2)).gridsearch(Xb, yb, lam=[grid_b, grid_b, grid_b], progress=False)

# Ordinary logistic regression: linear in x1 and x2, with dummies for g (C = inf: no penalty)
def linear_columns(Xb):
    return np.column_stack([Xb[:, 0], Xb[:, 1], Xb[:, 2] == 1, Xb[:, 2] == 2]).astype(float)
logit_lin = LogisticRegression(C=np.inf, max_iter=5000).fit(linear_columns(Xb), yb)


rows = []
for name, p in [("logistic GAM", gam_logit.predict_proba(Xb_test)),
                ("linear logistic regression", logit_lin.predict_proba(linear_columns(Xb_test))[:, 1])]:
    rows.append({"model": name, "test error rate": np.mean((p > 0.5) != yb_test),
                 "test log loss": log_loss(yb_test, p)})
display(pd.DataFrame(rows).set_index("model").round(4))
print("lambda for each term:", [float(v[0]) for v in gam_logit.lam])
print("EDoF of each term:", np.round(term_edof(gam_logit), 2))

plot_terms(gam_logit, Xb, [lambda t: 2 * f1_true(t), lambda t: 3 * (t - 0.5), GROUP_EFFECT], ylabel="log-odds term ")

**E8.** Which model predicts better on the test set? The linear logistic regression has the right
form for $x_2$ and $g$: what is it missing?

<details>
<summary>Show answer: E8</summary>

**E8. Answer.** The logistic GAM: test error rate 0.244 against 0.269, and log loss 0.505 against
0.542. The linear model has the right form for $x_2$ and $g$, but it treats $x_1$ as a straight line.
The true effect of $x_1$ is a sine wave with amplitude 2 on the log-odds scale, the largest effect in
the model, and a line can't follow it.

</details>

**E9.** Read the fitted $x_2$ term. By how much does the log-odds change when $x_2$ goes from 0 to
1, holding $x_1$ and $g$ fixed? By what factor does that multiply the odds of $Y = 1$? How does it
compare with the truth?

<details>
<summary>Show answer: E9</summary>

**E9. Answer.** The fitted line rises by about 3.2 on the log-odds scale as $x_2$ goes from 0 to 1,
holding $x_1$ and $g$ fixed. That multiplies the odds of $Y = 1$ by about $e^{3.2} \approx 24$. The
true slope is 3, a factor of $e^3 \approx 20$, so the estimate is close. The wide band near the ends
shows how uncertain the slope is.

</details>

---
## 5. Wrap-up questions

**Q1.** Suppose the true model is $y = x_1x_2 + \varepsilon$. What can a GAM with a smooth term for
each predictor fit, and what will it miss? How could you change the model to capture it? (pyGAM has a
`te()` term, and `mgcv` has `te()` and `s(x1, x2)`.)

<details>
<summary>Show answer: Q1</summary>

**Q1. Answer.** A GAM with a smooth term for each predictor can only fit $f_1(x_1) + f_2(x_2)$,
where the effect of $x_1$ is the same at every value of $x_2$. In $x_1x_2$ the effect of $x_1$ depends on
$x_2$. The GAM finds the best additive approximation, which is the part of $x_1x_2$ that looks like a
main effect in each predictor, and it misses the interaction. To capture it, add an interaction term:
the product $x_1x_2$ as a linear term, or a smooth surface in $x_1$ and $x_2$ together (`te(0, 1)` in
pyGAM; `te(x1, x2)` or `s(x1, x2)` in `mgcv`). ISL 7.7.1 discusses this as one of the limitations of
GAMs.

</details>

**Q2.** A colleague fits a GAM with a smooth term for every numeric predictor and uses one shared
$\lambda$, "to treat all the predictors fairly". What would you tell them?

<details>
<summary>Show answer: Q2</summary>

**Q2. Answer.** One shared $\lambda$ doesn't treat the predictors fairly, because the same $\lambda$
gives different terms different amounts of smoothing. In Section 1, $\lambda = 0.6$ gave the two smooth
terms 12.2 and 10.7 EDoF, and T1 showed that the amount also changes with $n$. More importantly, the
predictors' true effects differ: one may be strongly curved and another a straight line. In Section
3.3 the shared $\lambda$ left $x_2$ with 7.5 EDoF for a straight line, and it had a higher test MSE
than a separate $\lambda_j$ for each term. Give each term its own $\lambda_j$ and choose them from the
data: by GCV or cross-validation in pyGAM, or by REML in `mgcv`, which does it automatically.

</details>